# Chapter 29 Demo: From Wishlist to Working System
## Whiteboard Agent Capabilities and Iterative Architecture

**Core Claim:** Agent architecture is discovered iteratively, not designed upfront. The whiteboard exercise — list capabilities, group by data source, derive agents — consistently produces better designs than top-down specification.

**This notebook demonstrates:**
1. A simulated multi-agent system with measurable inter-agent communication
2. Top-down (by role) vs. Whiteboard (by data affinity) decomposition
3. An AI scaffold that proposes clusters — with a **Mandatory Human Decision Node**
4. A triggerable failure case: coupling density explosion under load

---

## Part 1: Flat Capability List (The Whiteboard)

In [1]:
import random
import statistics
from dataclasses import dataclass, field
from collections import defaultdict

# ============================================================
# CAPABILITY DEFINITIONS
# Each capability: name + data sources it requires.
# This is the "flat list on the whiteboard" — Step 1.
# ============================================================

CAPABILITIES = {
    "lookup_account":     {"name": "Look up customer account",     "data_sources": {"CRM"}},
    "check_subscription": {"name": "Check subscription status",    "data_sources": {"CRM", "Billing"}},
    "check_billing":      {"name": "Check billing history",        "data_sources": {"Billing"}},
    "search_kb":          {"name": "Search knowledge base",        "data_sources": {"KnowledgeBase"}},
    "draft_email":        {"name": "Draft email response",         "data_sources": {"KnowledgeBase", "CRM"}},
    "detect_urgency":     {"name": "Detect ticket urgency",        "data_sources": {"CRM", "TicketQueue"}},
    "route_to_human":     {"name": "Route to human agent",         "data_sources": {"TicketQueue", "StaffDirectory"}},
    "monitor_sla":        {"name": "Monitor SLA compliance",       "data_sources": {"TicketQueue", "SLAConfig"}},
    "generate_trends":    {"name": "Generate trend summaries",     "data_sources": {"TicketQueue", "CRM"}},
    "update_crm":         {"name": "Update CRM records",           "data_sources": {"CRM"}},
}

print("=" * 60)
print("STEP 1: Flat Capability List (The Whiteboard)")
print("=" * 60)
for cap_id, cap in CAPABILITIES.items():
    sources = ", ".join(sorted(cap["data_sources"]))
    print(f"  {cap['name']:<35} -> [{sources}]")
print(f"\nTotal capabilities: {len(CAPABILITIES)}")
all_sources = sorted(set().union(*(c['data_sources'] for c in CAPABILITIES.values())))
print(f"Unique data sources: {all_sources}")

STEP 1: Flat Capability List (The Whiteboard)
  Look up customer account            -> [CRM]
  Check subscription status           -> [Billing, CRM]
  Check billing history               -> [Billing]
  Search knowledge base               -> [KnowledgeBase]
  Draft email response                -> [CRM, KnowledgeBase]
  Detect ticket urgency               -> [CRM, TicketQueue]
  Route to human agent                -> [StaffDirectory, TicketQueue]
  Monitor SLA compliance              -> [SLAConfig, TicketQueue]
  Generate trend summaries            -> [CRM, TicketQueue]
  Update CRM records                  -> [CRM]

Total capabilities: 10
Unique data sources: ['Billing', 'CRM', 'KnowledgeBase', 'SLAConfig', 'StaffDirectory', 'TicketQueue']


## Part 2: Two Decomposition Strategies

In [2]:
# ============================================================
# ARCHITECTURE DEFINITIONS
# ============================================================

# Top-Down: grouped by job role (the intuitive approach)
TOPDOWN_AGENTS = {
    "Support": ["lookup_account", "search_kb", "draft_email", "check_subscription"],
    "Escalation": ["detect_urgency", "route_to_human", "monitor_sla"],
    "Analytics": ["check_billing", "generate_trends", "update_crm"],
}

# Whiteboard: grouped by data affinity
# Key insight: draft_email needs BOTH CRM and KnowledgeBase.
# search_kb is always called right before draft_email in task flows.
# So search_kb belongs with the CRM cluster, not in a separate agent.
WHITEBOARD_AGENTS = {
    "CRM_Hub": ["lookup_account", "check_subscription", "update_crm",
                "draft_email", "detect_urgency", "search_kb"],
    "TicketOps": ["route_to_human", "monitor_sla", "generate_trends"],
    "BillingOps": ["check_billing"],
}

def print_architecture(name, agents):
    print(f"\n{'=' * 60}")
    print(f"Architecture: {name}")
    print(f"{'=' * 60}")
    for agent_name, caps in agents.items():
        cap_names = [CAPABILITIES[c]["name"] for c in caps]
        all_src = set()
        for c in caps:
            all_src |= CAPABILITIES[c]["data_sources"]
        print(f"\n  Agent: {agent_name}")
        print(f"  Data sources: {sorted(all_src)}")
        for cn in cap_names:
            print(f"    - {cn}")

print_architecture("Top-Down (by Role)", TOPDOWN_AGENTS)
print_architecture("Whiteboard (by Data Affinity)", WHITEBOARD_AGENTS)


Architecture: Top-Down (by Role)

  Agent: Support
  Data sources: ['Billing', 'CRM', 'KnowledgeBase']
    - Look up customer account
    - Search knowledge base
    - Draft email response
    - Check subscription status

  Agent: Escalation
  Data sources: ['CRM', 'SLAConfig', 'StaffDirectory', 'TicketQueue']
    - Detect ticket urgency
    - Route to human agent
    - Monitor SLA compliance

  Agent: Analytics
  Data sources: ['Billing', 'CRM', 'TicketQueue']
    - Check billing history
    - Generate trend summaries
    - Update CRM records

Architecture: Whiteboard (by Data Affinity)

  Agent: CRM_Hub
  Data sources: ['Billing', 'CRM', 'KnowledgeBase', 'TicketQueue']
    - Look up customer account
    - Check subscription status
    - Update CRM records
    - Draft email response
    - Detect ticket urgency
    - Search knowledge base

  Agent: TicketOps
  Data sources: ['CRM', 'SLAConfig', 'StaffDirectory', 'TicketQueue']
    - Route to human agent
    - Monitor SLA compliance
  

## Part 3: Simulation Engine

In [3]:
@dataclass
class SimResult:
    arch: str
    task: str
    total_invocations: int
    cross_agent_calls: int
    coupling_density: float
    latency_ms: float
    trace: list = field(default_factory=list)

INTERNAL_MS = 5     # internal call latency
CROSS_MS = 45       # cross-agent call latency
JITTER_MS = 3       # random jitter

def find_agent(cap_id, agents):
    for name, caps in agents.items():
        if cap_id in caps:
            return name
    raise ValueError(f"'{cap_id}' not assigned!")

def run_task(task_name, seq, agents, arch_name):
    """Simulate a task through an architecture, measuring coupling and latency."""
    cross = 0
    latency = 0.0
    trace = []
    prev = None
    for cap_id in seq:
        curr = find_agent(cap_id, agents)
        is_cross = (prev is not None and curr != prev)
        if is_cross:
            cross += 1
            ms = CROSS_MS + random.uniform(-JITTER_MS, JITTER_MS)
            trace.append(f"  X {cap_id:<25} [{prev} -> {curr}] CROSS ({ms:.0f}ms)")
        else:
            ms = INTERNAL_MS + random.uniform(-JITTER_MS, JITTER_MS)
            trace.append(f"  . {cap_id:<25} [{curr}] internal ({ms:.0f}ms)")
        latency += ms
        prev = curr
    n = len(seq)
    cd = cross / max(n - 1, 1)
    return SimResult(arch_name, task_name, n, cross, cd, latency, trace)

print("Engine loaded.")
print(f"  Internal: ~{INTERNAL_MS}ms | Cross-agent: ~{CROSS_MS}ms | Penalty: {CROSS_MS//INTERNAL_MS}x")

Engine loaded.
  Internal: ~5ms | Cross-agent: ~45ms | Penalty: 9x


## Part 4: Representative Tasks

In [4]:
TASKS = {
    "basic_support": {
        "desc": "Customer asks a how-to question",
        "seq": ["lookup_account", "search_kb", "draft_email", "update_crm"],
    },
    "escalation": {
        "desc": "Urgent ticket detected and escalated",
        "seq": ["lookup_account", "check_subscription", "detect_urgency", "route_to_human", "update_crm"],
    },
    "billing_inquiry": {
        "desc": "Customer disputes a charge",
        "seq": ["lookup_account", "check_billing", "check_subscription", "draft_email", "update_crm"],
    },
    "weekly_analytics": {
        "desc": "Generate weekly ops report",
        "seq": ["generate_trends", "monitor_sla", "lookup_account"],
    },
    "full_lifecycle": {
        "desc": "Complex ticket: full lifecycle",
        "seq": ["lookup_account", "search_kb", "draft_email", "detect_urgency",
                "route_to_human", "monitor_sla", "update_crm"],
    },
}

print("Defined Tasks:")
for tid, t in TASKS.items():
    print(f"  {tid}: {t['desc']}")
    print(f"    {' -> '.join(t['seq'])}")

Defined Tasks:
  basic_support: Customer asks a how-to question
    lookup_account -> search_kb -> draft_email -> update_crm
  escalation: Urgent ticket detected and escalated
    lookup_account -> check_subscription -> detect_urgency -> route_to_human -> update_crm
  billing_inquiry: Customer disputes a charge
    lookup_account -> check_billing -> check_subscription -> draft_email -> update_crm
  weekly_analytics: Generate weekly ops report
    generate_trends -> monitor_sla -> lookup_account
  full_lifecycle: Complex ticket: full lifecycle
    lookup_account -> search_kb -> draft_email -> detect_urgency -> route_to_human -> monitor_sla -> update_crm


## Part 5: Head-to-Head Comparison

In [5]:
random.seed(42)
all_results = []

print("=" * 80)
print("HEAD-TO-HEAD: Top-Down vs. Whiteboard")
print("=" * 80)

for tid, t in TASKS.items():
    print(f"\n--- {t['desc']} ---")
    for aname, agents in [("Top-Down", TOPDOWN_AGENTS), ("Whiteboard", WHITEBOARD_AGENTS)]:
        r = run_task(tid, t["seq"], agents, aname)
        all_results.append(r)
        print(f"  [{aname}]")
        for line in r.trace:
            print(f"    {line}")
        print(f"    Coupling: {r.coupling_density:.2f} | Latency: {r.latency_ms:.0f}ms")

HEAD-TO-HEAD: Top-Down vs. Whiteboard

--- Customer asks a how-to question ---
  [Top-Down]
      . lookup_account            [Support] internal (6ms)
      . search_kb                 [Support] internal (2ms)
      . draft_email               [Support] internal (4ms)
      X update_crm                [Support -> Analytics] CROSS (43ms)
    Coupling: 0.33 | Latency: 55ms
  [Whiteboard]
      . lookup_account            [CRM_Hub] internal (6ms)
      . search_kb                 [CRM_Hub] internal (6ms)
      . draft_email               [CRM_Hub] internal (7ms)
      . update_crm                [CRM_Hub] internal (3ms)
    Coupling: 0.00 | Latency: 22ms

--- Urgent ticket detected and escalated ---
  [Top-Down]
      . lookup_account            [Support] internal (5ms)
      . check_subscription        [Support] internal (2ms)
      X detect_urgency            [Support -> Escalation] CROSS (43ms)
      . route_to_human            [Escalation] internal (5ms)
      X update_crm            

In [6]:
# ============================================================
# AGGREGATE COMPARISON
# ============================================================

print("\n" + "=" * 60)
print("AGGREGATE RESULTS")
print("=" * 60)

for arch in ["Top-Down", "Whiteboard"]:
    rs = [r for r in all_results if r.arch == arch]
    ac = statistics.mean(r.coupling_density for r in rs)
    al = statistics.mean(r.latency_ms for r in rs)
    tc = sum(r.cross_agent_calls for r in rs)
    print(f"\n  {arch}:")
    print(f"    Avg coupling density: {ac:.2f}")
    print(f"    Avg task latency:     {al:.0f}ms")
    print(f"    Total cross-agent calls: {tc}")

td_lat = statistics.mean(r.latency_ms for r in all_results if r.arch == "Top-Down")
wb_lat = statistics.mean(r.latency_ms for r in all_results if r.arch == "Whiteboard")
td_coup = statistics.mean(r.coupling_density for r in all_results if r.arch == "Top-Down")
wb_coup = statistics.mean(r.coupling_density for r in all_results if r.arch == "Whiteboard")
improvement = (td_lat - wb_lat) / td_lat * 100

print(f"\n  Coupling reduction: {td_coup:.2f} -> {wb_coup:.2f}")
print(f"  Latency reduction:  {improvement:.0f}%")
print(f"\n  >> Architecture is the leverage point, not the model.")


AGGREGATE RESULTS

  Top-Down:
    Avg coupling density: 0.58
    Avg task latency:     104ms
    Total cross-agent calls: 10

  Whiteboard:
    Avg coupling density: 0.37
    Avg task latency:     77ms
    Total cross-agent calls: 7

  Coupling reduction: 0.58 -> 0.37
  Latency reduction:  26%

  >> Architecture is the leverage point, not the model.


## Part 6: AI Scaffold with Human Decision Node

The AI performs a **bounded enumeration task**: propose agent clusters from capability + data source tags.

**The AI makes a specific error** — grouping by semantic similarity (retrieval/action/analysis) instead of data affinity. The **Mandatory Human Decision Node** catches this.

In [7]:
# ============================================================
# AI SCAFFOLD: Simulated LLM Capability Clustering
# ============================================================
# In production, this would call an LLM API.
# Here we simulate the output to keep the notebook runnable.
# ============================================================

def ai_propose_clusters(caps):
    """Simulated LLM output.
    DELIBERATE ERROR: groups by semantic similarity instead of data affinity."""
    reasoning = """
    AI Reasoning (simulated LLM output):
    
    I'll organize these capabilities by functional purpose:
    
    - "RetrievalAgent": capabilities that FETCH information
      -> lookup_account, check_subscription, check_billing, search_kb
      
    - "ActionAgent": capabilities that DO things
      -> draft_email, route_to_human, update_crm
      
    - "AnalysisAgent": capabilities that ANALYZE or MONITOR
      -> detect_urgency, monitor_sla, generate_trends
    
    This gives clean separation of concerns: read, write, analyze.
    """
    proposed = {
        "RetrievalAgent": ["lookup_account", "check_subscription", "check_billing", "search_kb"],
        "ActionAgent": ["draft_email", "route_to_human", "update_crm"],
        "AnalysisAgent": ["detect_urgency", "monitor_sla", "generate_trends"],
    }
    return proposed, reasoning

ai_clusters, ai_reasoning = ai_propose_clusters(CAPABILITIES)
print("=" * 60)
print("AI SCAFFOLD: Proposed Clusters")
print("=" * 60)
print(ai_reasoning)
print_architecture("AI-Proposed (Semantic Grouping)", ai_clusters)

AI SCAFFOLD: Proposed Clusters

    AI Reasoning (simulated LLM output):
    
    I'll organize these capabilities by functional purpose:
    
    - "RetrievalAgent": capabilities that FETCH information
      -> lookup_account, check_subscription, check_billing, search_kb
      
    - "ActionAgent": capabilities that DO things
      -> draft_email, route_to_human, update_crm
      
    - "AnalysisAgent": capabilities that ANALYZE or MONITOR
      -> detect_urgency, monitor_sla, generate_trends
    
    This gives clean separation of concerns: read, write, analyze.
    

Architecture: AI-Proposed (Semantic Grouping)

  Agent: RetrievalAgent
  Data sources: ['Billing', 'CRM', 'KnowledgeBase']
    - Look up customer account
    - Check subscription status
    - Check billing history
    - Search knowledge base

  Agent: ActionAgent
  Data sources: ['CRM', 'KnowledgeBase', 'StaffDirectory', 'TicketQueue']
    - Draft email response
    - Route to human agent
    - Update CRM records

  Age

In [8]:
# Test AI proposal performance
random.seed(42)
ai_results = []
for tid, t in TASKS.items():
    r = run_task(tid, t["seq"], ai_clusters, "AI-Proposed")
    ai_results.append(r)
    print(f"  {t['desc']:<45} coupling={r.coupling_density:.2f}  latency={r.latency_ms:.0f}ms")

ai_coup = statistics.mean(r.coupling_density for r in ai_results)
ai_lat = statistics.mean(r.latency_ms for r in ai_results)

print(f"\n  AI-Proposed:  avg coupling = {ai_coup:.2f}")
print(f"  Top-Down:     avg coupling = {td_coup:.2f}")
print(f"  Whiteboard:   avg coupling = {wb_coup:.2f}")
print(f"\n  >> AI semantic grouping is worse than the whiteboard method.")
print(f"     Better than pure intuition, but still draws wrong boundaries.")

  Customer asks a how-to question               coupling=0.33  latency=55ms
  Urgent ticket detected and escalated          coupling=0.50  latency=107ms
  Customer disputes a charge                    coupling=0.25  latency=56ms
  Generate weekly ops report                    coupling=0.50  latency=54ms
  Complex ticket: full lifecycle                coupling=0.83  latency=234ms

  AI-Proposed:  avg coupling = 0.48
  Top-Down:     avg coupling = 0.58
  Whiteboard:   avg coupling = 0.37

  >> AI semantic grouping is worse than the whiteboard method.
     Better than pure intuition, but still draws wrong boundaries.


In [9]:
# ============================================================
# MANDATORY HUMAN DECISION NODE
# ============================================================
#
# The AI proposed grouping by SEMANTIC FUNCTION
# (retrieval / action / analysis).
#
# PROBLEM IDENTIFIED:
# The AI grouped lookup_account (CRM) with search_kb
# (KnowledgeBase) because both are "retrieval" operations.
# But draft_email needs BOTH CRM and KB data — in real task
# flows, search_kb is always called right before draft_email.
# Separating them forces a cross-agent round-trip every time.
#
# The AI also split update_crm into "ActionAgent" away from
# lookup_account in "RetrievalAgent" — but every task that
# reads CRM also writes CRM at the end. Splitting read/write
# of the SAME data source across agents is the classic error.
#
# DECISION: REJECTED
# REASON: Semantic similarity != data affinity. Grouping by
# "retrieval vs action" splits CRM read/write across agents.
#
# CORRECTION: Use data-affinity clustering (whiteboard method).
# ============================================================

human_decision = "REJECTED"
rejection_reason = (
    "AI grouped by semantic function (retrieval/action/analysis) "
    "instead of data source affinity. This splits CRM read and "
    "write operations across agents (lookup_account in Retrieval, "
    "update_crm in Action), forcing a cross-agent call at the end "
    "of every customer task. It also separates search_kb from "
    "draft_email despite them always being called in sequence."
)

corrected_clusters = WHITEBOARD_AGENTS

print("=" * 60)
print("HUMAN DECISION NODE")
print("=" * 60)
print(f"\n  Decision: {human_decision}")
print(f"\n  Reason: {rejection_reason}")
print(f"\n  Performance comparison:")
print(f"    AI-Proposed coupling:  {ai_coup:.2f}")
print(f"    Top-Down coupling:     {td_coup:.2f}")
print(f"    Whiteboard coupling:   {wb_coup:.2f}  <-- best")
print(f"\n  >> Human saw 'CRM read + CRM write' as the real boundary.")
print(f"     AI saw 'retrieval vs action' as a category.")
print(f"     Data affinity > semantic similarity.")

HUMAN DECISION NODE

  Decision: REJECTED

  Reason: AI grouped by semantic function (retrieval/action/analysis) instead of data source affinity. This splits CRM read and write operations across agents (lookup_account in Retrieval, update_crm in Action), forcing a cross-agent call at the end of every customer task. It also separates search_kb from draft_email despite them always being called in sequence.

  Performance comparison:
    AI-Proposed coupling:  0.48
    Top-Down coupling:     0.58
    Whiteboard coupling:   0.37  <-- best

  >> Human saw 'CRM read + CRM write' as the real boundary.
     AI saw 'retrieval vs action' as a category.
     Data affinity > semantic similarity.


## Part 7: Triggerable Failure Case — Progressive Boundary Degradation

We **deliberately break** the whiteboard architecture by progressively moving CRM-affine capabilities out of the CRM_Hub into other agents.

### Exercise for the Reader:
Modify `level_3` to create your own worst-case boundary split, then run the benchmark.

In [10]:
# ============================================================
# FAILURE CASE: Progressive Boundary Degradation
# ============================================================
# We move CRM-affine capabilities OUT of the CRM_Hub
# to simulate what happens with wrong boundaries.
# ============================================================

# Level 0: Whiteboard (correct)
level_0 = {
    "A": ["lookup_account", "check_subscription", "update_crm", "draft_email", "detect_urgency", "search_kb"],
    "B": ["route_to_human", "monitor_sla", "generate_trends"],
    "C": ["check_billing"],
}

# Level 1: Move detect_urgency out of CRM cluster
level_1 = {
    "A": ["lookup_account", "check_subscription", "update_crm", "draft_email", "search_kb"],
    "B": ["detect_urgency", "route_to_human", "monitor_sla", "generate_trends"],
    "C": ["check_billing"],
}

# Level 2: Also move draft_email + search_kb out
level_2 = {
    "A": ["lookup_account", "check_subscription", "update_crm"],
    "B": ["detect_urgency", "route_to_human", "monitor_sla", "generate_trends"],
    "C": ["check_billing", "draft_email", "search_kb"],
}

# Level 3: Maximum fragmentation - CRM caps distributed evenly
level_3 = {
    "A": ["lookup_account", "search_kb", "monitor_sla"],
    "B": ["check_subscription", "draft_email", "route_to_human"],
    "C": ["check_billing", "detect_urgency", "update_crm", "generate_trends"],
}

levels = [
    ("Level 0: Whiteboard (correct)", level_0),
    ("Level 1: 1 CRM cap moved out", level_1),
    ("Level 2: 3 CRM caps moved out", level_2),
    ("Level 3: Max fragmentation", level_3),
]

print("=" * 70)
print("FAILURE CASE: Progressive Boundary Degradation")
print("=" * 70)
print("\nMoving CRM-affine capabilities OUT of the CRM cluster...\n")

level_metrics = []
for lname, agents in levels:
    random.seed(42)
    rs = [run_task(tid, t["seq"], agents, lname) for tid, t in TASKS.items()]
    ac = statistics.mean(r.coupling_density for r in rs)
    al = statistics.mean(r.latency_ms for r in rs)
    tc = sum(r.cross_agent_calls for r in rs)
    level_metrics.append((lname, ac, al, tc))
    print(f"  {lname:<35} coupling={ac:.2f}  latency={al:.0f}ms  cross_calls={tc}")

baseline = level_metrics[0][2]
print(f"\n  Latency degradation (vs Level 0):")
for name, coup, lat, _ in level_metrics:
    mult = lat / baseline
    bar = "#" * int(mult * 15)
    print(f"    {name:<35} {mult:.1f}x  {bar}")

print(f"\n  >> Same capabilities. Same model. Different boundaries.")
print(f"     Coupling density drives latency. Architecture is the variable.")

FAILURE CASE: Progressive Boundary Degradation

Moving CRM-affine capabilities OUT of the CRM cluster...

  Level 0: Whiteboard (correct)       coupling=0.37  latency=77ms  cross_calls=7
  Level 1: 1 CRM cap moved out        coupling=0.37  latency=77ms  cross_calls=7
  Level 2: 3 CRM caps moved out       coupling=0.63  latency=117ms  cross_calls=12
  Level 3: Max fragmentation          coupling=0.75  latency=141ms  cross_calls=15

  Latency degradation (vs Level 0):
    Level 0: Whiteboard (correct)       1.0x  ###############
    Level 1: 1 CRM cap moved out        1.0x  ###############
    Level 2: 3 CRM caps moved out       1.5x  ######################
    Level 3: Max fragmentation          1.8x  ###########################

  >> Same capabilities. Same model. Different boundaries.
     Coupling density drives latency. Architecture is the variable.


In [11]:
# ============================================================
# LOAD TEST: 100 Concurrent Tasks
# Cross-agent calls compete for the message queue.
# ============================================================

NUM_TASKS = 100
QUEUE_PENALTY_MS = 8  # per cross-agent call under contention

def load_test(agents, name, n):
    random.seed(42)
    tids = list(TASKS.keys())
    lats = []
    total_cross = 0
    for i in range(n):
        tid = random.choice(tids)
        r = run_task(tid, TASKS[tid]["seq"], agents, name)
        # Queue penalty grows as more cross-agent calls accumulate
        qp = r.cross_agent_calls * QUEUE_PENALTY_MS * (total_cross / max(n, 1))
        lats.append(r.latency_ms + qp)
        total_cross += r.cross_agent_calls
    lats.sort()
    return {
        "p50": lats[int(n*0.50)],
        "p95": lats[int(n*0.95)],
        "p99": lats[int(n*0.99)],
        "cross": total_cross,
    }

print(f"{'=' * 70}")
print(f"LOAD TEST: {NUM_TASKS} Concurrent Tasks")
print(f"{'=' * 70}")
print(f"Queue contention: +{QUEUE_PENALTY_MS}ms per cross-agent call\n")

for lname, agents in levels:
    m = load_test(agents, lname, NUM_TASKS)
    print(f"  {lname}")
    print(f"    p50={m['p50']:.0f}ms  p95={m['p95']:.0f}ms  p99={m['p99']:.0f}ms  total_cross={m['cross']}")
    print()

print("  >> Under load, boundary errors compound nonlinearly.")
print("     p95 latency explodes at higher fragmentation levels.")
print("     No model upgrade fixes this. Only redrawing boundaries does.")

LOAD TEST: 100 Concurrent Tasks
Queue contention: +8ms per cross-agent call

  Level 0: Whiteboard (correct)
    p50=108ms  p95=136ms  p99=142ms  total_cross=135

  Level 1: 1 CRM cap moved out
    p50=108ms  p95=136ms  p99=142ms  total_cross=135

  Level 2: 3 CRM caps moved out
    p50=130ms  p95=233ms  p99=261ms  total_cross=237

  Level 3: Max fragmentation
    p50=162ms  p95=337ms  p99=355ms  total_cross=302

  >> Under load, boundary errors compound nonlinearly.
     p95 latency explodes at higher fragmentation levels.
     No model upgrade fixes this. Only redrawing boundaries does.


## Part 8: Whiteboard Algorithm (Reusable)

In [12]:
def whiteboard_decompose(capabilities, max_per_agent=7):
    """Automated whiteboard method.
    1. Input: flat capability list with data source tags
    2. Cluster each capability by its most-connected data source
    3. Merge tiny clusters (< 2 caps) into nearest neighbor
    Returns: dict mapping agent name -> list of capability IDs
    """
    # Group by most-connected data source
    src_map = defaultdict(list)
    for cid, c in capabilities.items():
        best_src, best_n = None, 0
        for src in c["data_sources"]:
            n = sum(1 for o in capabilities.values() if src in o["data_sources"])
            if n > best_n:
                best_n = n
                best_src = src
        src_map[best_src].append(cid)

    clusters = dict(src_map)

    # Merge tiny clusters into nearest neighbor by source overlap
    smalls = [s for s, cs in clusters.items() if len(cs) < 2]
    for s in smalls:
        s_srcs = set()
        for cid in clusters[s]:
            s_srcs |= capabilities[cid]["data_sources"]
        best_merge, best_ov = None, 0
        for o, ocs in clusters.items():
            if o == s or o in smalls:
                continue
            o_srcs = set()
            for cid in ocs:
                o_srcs |= capabilities[cid]["data_sources"]
            ov = len(s_srcs & o_srcs)
            if ov > best_ov:
                best_ov, best_merge = ov, o
        if best_merge:
            clusters[best_merge].extend(clusters[s])
            del clusters[s]

    return {f"Agent_{src}": cs for src, cs in
            sorted(clusters.items(), key=lambda x: -len(x[1]))}

# Run the algorithm
auto = whiteboard_decompose(CAPABILITIES)
print_architecture("Auto-Whiteboard", auto)

# Validate
random.seed(42)
auto_rs = [run_task(tid, t["seq"], auto, "Auto") for tid, t in TASKS.items()]
auto_c = statistics.mean(r.coupling_density for r in auto_rs)
auto_l = statistics.mean(r.latency_ms for r in auto_rs)
print(f"\n  Auto-whiteboard: coupling={auto_c:.2f}, latency={auto_l:.0f}ms")
print(f"  Top-Down:        coupling={td_coup:.2f}, latency={td_lat:.0f}ms")
print(f"  Whiteboard:      coupling={wb_coup:.2f}, latency={wb_lat:.0f}ms")


Architecture: Auto-Whiteboard

  Agent: Agent_CRM
  Data sources: ['Billing', 'CRM', 'KnowledgeBase', 'TicketQueue']
    - Look up customer account
    - Check subscription status
    - Draft email response
    - Detect ticket urgency
    - Generate trend summaries
    - Update CRM records
    - Check billing history
    - Search knowledge base

  Agent: Agent_TicketQueue
  Data sources: ['SLAConfig', 'StaffDirectory', 'TicketQueue']
    - Route to human agent
    - Monitor SLA compliance

  Auto-whiteboard: coupling=0.37, latency=69ms
  Top-Down:        coupling=0.58, latency=104ms
  Whiteboard:      coupling=0.37, latency=77ms


## Summary

| Architecture | Avg Coupling Density | Total Cross-Agent Calls | Method |
|---|---|---|---|
| **Whiteboard** (data affinity) | **~0.37** | **7** | Bottom-up, data-driven |
| AI-Proposed (semantic) | ~0.48 | 10 | LLM pattern-matching |
| Top-Down (by role) | ~0.58 | 10 | Intuition / org chart |

### Key Findings:

1. **Top-down decomposition** produces the highest coupling — it draws boundaries around job descriptions, not data flow.
2. **AI-proposed decomposition** improves over intuition but still makes errors — LLMs optimize for conceptual cleanliness, not communication cost.
3. **The whiteboard method** produces the lowest coupling by using data affinity as the clustering criterion.
4. **Under load**, coupling density differences become catastrophic due to queue contention compounding.
5. **No model change fixes a boundary error.** The failure is architectural. Only redrawing the agent graph fixes it.

### The Master Argument, Demonstrated:

**Architecture is the leverage point, not the model.** Same capabilities, same tasks, same (simulated) model. The only variable was where we drew the agent boundaries. That single architectural decision determined whether the system scales or collapses.